*Recommendation method 3*

Machine Learning Prediction

In [1]:
import pandas as pd 
import numpy as np

In [94]:
df = pd.read_csv("/Users/mac/Desktop/personalisation/personalisation-23-24-main/mini/data/animes.csv")
df = df.dropna()
df.head(5)

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,img_url,link
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...","Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,https://cdn.myanimelist.net/images/anime/9/766...,https://myanimelist.net/anime/28891/Haikyuu_Se...
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...","Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,https://cdn.myanimelist.net/images/anime/3/671...,https://myanimelist.net/anime/23273/Shigatsu_w...
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...","Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,https://cdn.myanimelist.net/images/anime/6/867...,https://myanimelist.net/anime/34599/Made_in_Abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...","Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,https://cdn.myanimelist.net/images/anime/1223/...,https://myanimelist.net/anime/5114/Fullmetal_A...
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']","Jan 6, 2017",1.0,214621,502,22.0,8.83,https://cdn.myanimelist.net/images/anime/3/815...,https://myanimelist.net/anime/31758/Kizumonoga...


In [95]:
df["score"].value_counts()

score
6.13    123
6.11    111
6.10    101
6.09    101
6.14     96
       ... 
3.10      1
3.12      1
3.00      1
2.52      1
4.14      1
Name: count, Length: 572, dtype: int64

In [96]:
len(df), len(df["uid"].unique()), len(df["ranked"].unique())

(15187, 13677, 13554)

In [97]:
import pandas as pd

df['aired'] = df['aired'].str.split(' to ').str[0]
df['aired'] = pd.to_datetime(df['aired'], format='%b %d, %Y', errors='coerce')

df['aired'] = df['aired'].dt.strftime('%B %d, %Y')

print(df)
print(df.dtypes)


         uid                                        title  \
0      28891                      Haikyuu!! Second Season   
1      23273                      Shigatsu wa Kimi no Uso   
2      34599                                Made in Abyss   
3       5114             Fullmetal Alchemist: Brotherhood   
4      31758             Kizumonogatari III: Reiketsu-hen   
...      ...                                          ...   
19306  32979                                Flip Flappers   
19307    123                                Fushigi Yuugi   
19308   1281                             Gakkou no Kaidan   
19309    450  InuYasha Movie 2: Kagami no Naka no Mugenjo   
19310     87     Mobile Suit Gundam: Char's Counterattack   

                                                synopsis  \
0      Following their participation at the Inter-Hig...   
1      Music accompanies the path of the human metron...   
2      The Abyss—a gaping chasm stretching down into ...   
3      "In order for someth

In [98]:
df["aired"] = pd.to_datetime(df["aired"])
df_cleaned = df[~df['aired'].isna()]
df["aired"]

0       2015-10-04
1       2014-10-10
2       2017-07-07
3       2009-04-05
4       2017-01-06
           ...    
19306   2016-10-06
19307   1995-04-06
19308   2000-10-22
19309   2002-12-21
19310   1988-03-12
Name: aired, Length: 15187, dtype: datetime64[ns]

In [99]:
df.dtypes

uid                    int64
title                 object
synopsis              object
genre                 object
aired         datetime64[ns]
episodes             float64
members                int64
popularity             int64
ranked               float64
score                float64
img_url               object
link                  object
dtype: object

In [100]:
df = df.sort_values(by="aired")

In [101]:
uids = df["uid"].unique()

In [102]:
data = []
last_date = df["aired"].max()
df['days_since_end'] = (last_date - df['aired']).dt.days

grouped = df.groupby('uid').agg({'ranked': list, 'days_since_end': list, 'score': list})

In [116]:
df['days_since_end']

8061     37476.0
3501     37435.0
8097     37378.0
9297     37219.0
5020     37172.0
          ...   
18946        NaN
19010        NaN
19034        NaN
19077        NaN
19130        NaN
Name: days_since_end, Length: 15187, dtype: float64

In [103]:
data = pd.DataFrame(grouped)
data_cleaned = data.dropna()
data

,ranked,days_since_end,score
uid,,,
1,"[26.0, 26.0]","[7938.0, 7938.0]","[8.81, 8.81]"
5,"[149.0, 149.0]","[6691.0, 6691.0]","[8.4, 8.4]"
6,"[256.0, 256.0]","[7940.0, 7940.0]","[8.28, 8.28]"
7,[2487.0],[6387.0],[7.32]
8,[3704.0],[5566.0],[7.02]
...,...,...,...
40711,[10196.0],[44.0],[5.39]
40769,[9633.0],[26.0],[5.59]
40780,[9686.0],[26.0],[5.58]


In [104]:
import torch

In [114]:
items = np.array(uids["ranked"])
items

array([9340.])

In [115]:
days = np.array(uids["days_since_end"])
days

array([16.])

In [112]:
max_len = 10
X_list = []
y_list = []
sequence_lengths = []
for _, item in data.iterrows():
    items = np.array(item["ranked"])
    days = np.array(item["days_since_end"])
    ratings = np.array(item["score"])
    if len(items) >= max_len:
        items_tensor = torch.tensor(items[:-1], dtype=torch.int)
        days_tensor = torch.tensor(days[:-1], dtype=torch.int)
        ratings_tensor = torch.tensor(ratings[:-1], dtype=torch.int)
        
        num_sequences = len(items_tensor) - max_len + 1
        if num_sequences < 1:
            num_sequences = 1
            
        sequence_lengths.append(num_sequences)
        for start_idx in range(num_sequences):
            end_idx = start_idx + max_len
            items_sequence = items_tensor[start_idx:end_idx]
            days_sequence = days_tensor[start_idx:end_idx]
            ratings_sequence = ratings_tensor[start_idx:end_idx]
            if items_sequence.shape[0] < max_len:
                pad_size = max_len - items_sequence.shape[0]
                items_sequence = torch.nn.functional.pad(items_sequence, (0, pad_size), "constant", 0)
                days_sequence = torch.nn.functional.pad(days_sequence, (0, pad_size), "constant", 0)
                ratings_sequence = torch.nn.functional.pad(ratings_sequence, (0, pad_size), "constant", 0)
                print(ratings_sequence)

            sequence = torch.cat((items_sequence, days_sequence, ratings_sequence), dim=0)
            X_list.append(sequence)

            target_items = items_tensor[start_idx+1:end_idx+1]
            if target_items.shape[0] < max_len:
                target_items = torch.nn.functional.pad(target_items, (0, max_len - target_items.shape[0]), "constant", 0)
            y_list.append(target_items)

X_list.insert(0, torch.tensor([0]))
y_list.insert(0, torch.tensor([0]))

X = torch.stack(X_list).view(len(X_list), -1)
y = torch.stack(y_list).view(len(y_list), -1)
print(X.shape, y.shape)


torch.Size([1, 1]) torch.Size([1, 1])


In [ ]:
all_input_items = X[:,0:max_len].flatten()
all_target_items = y.flatten()
all_items = np.concatenate((all_input_items, all_target_items))
unique_items = np.unique(all_items)

item_to_index = {item_id:i for i, item_id in enumerate(unique_items)}
index_to_item = {v:k for k,v in item_to_index.items()}    

item_indexes = np.array([item_to_index[i.item()] for i in all_input_items]).reshape((len(X),max_len))
X[:,0:max_len] = torch.IntTensor(item_indexes)

target_indexes = np.array([item_to_index[i.item()] for i in all_target_items]).reshape((len(y),max_len))
y = torch.IntTensor(target_indexes)

In [ ]:
days = X[:, max_len:]
days_max = days.max()

boundaries = torch.linspace(0, days_max, steps=101)  
boundaries = boundaries.to(days.device) 
bin_indices = torch.searchsorted(boundaries, days, right=True) - 1
bin_indices = torch.clamp(bin_indices, 0, 99) 
X[:, max_len:] = bin_indices

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

class RunwayDataset(Dataset):
    def __init__(self, features, labels):
        self.X = features
        self.y = labels
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
from torch.utils.data import random_split
total = len(X)
all_indexes = np.arange(total)
np.random.shuffle(all_indexes)
device = torch.device("mps")

split = 0.8
train_indexes = all_indexes[:int(total*split)]
val_indexes = all_indexes[int((total*split)):]
train_x = X[train_indexes].to(device)
val_x = X[val_indexes].to(device)

train_y = y[train_indexes].to(device)
val_y = y[val_indexes].to(device)

train_dataset = RunwayDataset(train_x,train_y)
validation_dataset = RunwayDataset(val_x,val_y)
torch.save(train_dataset, "train_dataset.pt")
torch.save(validation_dataset, "val_dataset.pt")

In [ ]:
train_dataset = torch.load('train_dataset.pt')
validation_dataset = torch.load('val_dataset.pt')

In [ ]:
class RunwayRecommender(torch.nn.Module):
    
    def __init__(self, num_items, num_days, embedding_size=20, hidden_size=64, seq_len=5):
        super().__init__()

        self.embedding_size = embedding_size
        self.days_embedding = torch.nn.Embedding(num_days, embedding_size)
        self.item_embedding = torch.nn.Embedding(num_items, embedding_size)
        
        self.position = torch.arange(0,seq_len).to(device)

        self.batch_norm1 = torch.nn.BatchNorm1d(seq_len)
        
        self.lstm1 = torch.nn.LSTM(
            input_size=embedding_size + embedding_size,
            hidden_size=hidden_size,
        )
        
        self.batch_norm2 = torch.nn.BatchNorm1d(seq_len)
        
        self.att = torch.nn.MultiheadAttention(
                embed_dim=hidden_size,
                num_heads=1,
                dropout=0.3
        )
        self.out_fc = torch.nn.Linear(hidden_size, num_items)
        
    
    def forward(self, inputs):
        
        seq_length = int(np.floor(inputs.shape[1]/2))
        
        items = inputs[:,:seq_length]
        days = inputs[:,seq_length:]
        
        embedded_items = self.item_embedding(items)
        embedded_days = self.days_embedding(days)
        
        concat_embedding_input = torch.cat((embedded_items, embedded_days), dim=-1)
        concat_embedding_input = self.batch_norm1(concat_embedding_input)
        
        rnn, _ = self.lstm1(concat_embedding_input)
        rnn = self.batch_norm2(rnn)

        rnn = rnn.permute(1, 0, 2)
        att, _ = self.att(rnn, rnn, rnn)
        
        att = att.permute(1, 0, 2)
        output = self.out_fc(att)
        
        return output

In [ ]:
EMBEDDING_SIZE = 50
num_items = len(unique_items)

model = RunwayRecommender(num_items, 101, EMBEDDING_SIZE, 32, max_len).to(device)

train_dl = DataLoader(train_dataset, batch_size=512, shuffle=True)
validation_dl = DataLoader(validation_dataset, batch_size=512, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
from torch.nn.functional import softmax

def loss_fn(pred, real):
    
    seq_len = pred.shape[1]
    total = 0
    for i in range(seq_len):
        p = pred[:,i]
        r = real[:,i]
        loss = torch.nn.CrossEntropyLoss(reduction='none')(p,r)
        mask = torch.logical_not(torch.eq(r, 0))
        mask = mask.type_as(loss)
        loss *= mask
        total += torch.mean(loss)
        
    return total/seq_len

In [ ]:
epochs = 300
for i in range(epochs):
    
    def run_epoch(dl, train=True):
        
        running_loss = 0.0
        matches = 0
        total = 0
        model.train(train)    
        
        for index, batch in enumerate(dl):

            inputs, labels = batch
            labels = labels.long()
            total += labels.size(0)
            
            model.zero_grad()
            outputs = model(inputs)

            k = 15
            predicted = torch.topk(softmax(outputs,dim=-1), k, dim =-1).indices
            topk_matches = [1 for i,row in enumerate(predicted[:,-1,:]) if labels[i,-1] in row]
            matches += np.array(topk_matches).sum()
            
            loss = loss_fn(outputs, labels)
            running_loss += loss
            if train:
                loss.backward()
                optimizer.step()
        
        acc = (matches / total) * 100
        avg_loss = running_loss / (index + 1)
        return acc, avg_loss
    
    acc, avg_loss = run_epoch(train_dl)
    with torch.no_grad():
        vacc, avg_vloss = run_epoch(validation_dl, False)
    
    print('Epoch {} Loss {:0.3f} Val Loss {:0.3f} Acc {:0.3f}% Val Acc {:0.3f}%'.format(i, avg_loss, avg_vloss, acc, vacc))

In [ ]:
torch.save(model.state_dict(), 'model.pt')

model = RunwayRecommender(num_items, 101, EMBEDDING_SIZE, 32, max_len).to(device)
model.load_state_dict(torch.load('model.pt'))

In [ ]:
import random
for batch in validation_dl:
    random_index = random.randint(0, len(batch) - 1)
    random_item = batch[0][random_index].unsqueeze(0)
    print(random_item)
    outputs = model(random_item)[:,-1].squeeze(0)
    print(outputs.shape)
    break 

predicted = torch.argmax(softmax(outputs)).item()
id_in_catalogue = index_to_item[predicted]
print("sequence", random_item.cpu().numpy())
print("recommend",id_in_catalogue)
df[df["title"] == id_in_catalogue]